<a href="https://colab.research.google.com/github/pksX01/AI-practice/blob/main/Gen%20AI/Langchain_openai_pyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.4.1-py2.py3-none-any.whl size=311285387 sha256=f3d571cc4ca11b2e2e2e0ed3a47bf0b1f37f243c92e612e3e931d17fa8f43d8b
  Stored in directory: /root/.cache/pip/wheels/0d/77/a3/ff2f74cc9ab41f8f594dabf0579c2a7c6de920d584206e0834
Successfully built pyspark


In [ ]:
!pip install langchain

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 25.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 5.8 MB/s eta 0:00:00


In [ ]:
!pip install openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.6 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving healthcare-dataset-stroke-data.csv to healthcare-dataset-stroke-data.csv


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [ ]:
schema = 'langchain_spark'
tbl_nm = 'stroke_data'
spark.sql(f"CREATE DATABASE IF NOT EXISTS {schema}")
df = spark.read.csv('/content/healthcare-dataset-stroke-data.csv', header=True, inferSchema=True)
df.write.saveAsTable(f"{schema}.{tbl_nm}")

In [ ]:
spark.table(f"{schema}.{tbl_nm}").show()

+-----+------+----+------------+-------------+------------+-------------+--------------+-----------------+----+---------------+------+
|   id|gender| age|hypertension|heart_disease|ever_married|    work_type|Residence_type|avg_glucose_level| bmi| smoking_status|stroke|
+-----+------+----+------------+-------------+------------+-------------+--------------+-----------------+----+---------------+------+
| 9046|  Male|67.0|           0|            1|         Yes|      Private|         Urban|           228.69|36.6|formerly smoked|     1|
|51676|Female|61.0|           0|            0|         Yes|Self-employed|         Rural|           202.21| N/A|   never smoked|     1|
|31112|  Male|80.0|           0|            1|         Yes|      Private|         Rural|           105.92|32.5|   never smoked|     1|
|60182|Female|49.0|           0|            0|         Yes|      Private|         Urban|           171.23|34.4|         smokes|     1|
| 1665|Female|79.0|           1|            0|         

In [ ]:
from langchain.agents import create_spark_sql_agent
from langchain.agents.agent_toolkits import SparkSQLToolkit
from langchain.chat_models import ChatOpenAI
from langchain.utilities import SparkSQL

In [ ]:
#spark.sql(f"drop table {schema}.{tbl_nm}")

In [ ]:
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
callback_manager = CallbackManager([StreamingStdOutCallbackHandler])

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ''

In [ ]:
llm = ChatOpenAI(temperature=0)

In [ ]:
spark_sql = SparkSQL(schema=schema)
toolkit = SparkSQLToolkit(db=spark_sql, llm=llm)
agent_executor = create_spark_sql_agent(llm=llm, toolkit=toolkit, verbose=True)

In [ ]:
agent_executor.run("How many persons have heart_disease?")



> Entering new AgentExecutor chain...
Action: list_tables_sql_db
Action Input: ""
Observation: stroke_data
Thought:I can query the "stroke_data" table to find the number of persons with heart disease.
Action: query_sql_db
Action Input: "SELECT COUNT(*) FROM stroke_data WHERE heart_disease = 1"
Observation: [('276',)]
Thought:There are 276 persons with heart disease.
Final Answer: 276

> Finished chain.


'276'

In [ ]:
df.filter(df['heart_disease'] == 1).count()

276

In [ ]:
agent_executor.run("How many persons are married?")



> Entering new AgentExecutor chain...
Action: list_tables_sql_db
Action Input: ""
Observation: stroke_data
Thought:I see that there is a table called "stroke_data" in the database. I should check the schema of this table to see if it has any columns related to marital status.
Action: schema_sql_db
Action Input: "stroke_data"
Observation: CREATE TABLE langchain_spark.stroke_data (
  id INT,
  gender STRING,
  age DOUBLE,
  hypertension INT,
  heart_disease INT,
  ever_married STRING,
  work_type STRING,
  Residence_type STRING,
  avg_glucose_level DOUBLE,
  bmi STRING,
  smoking_status STRING,
  stroke INT)
;

/*
3 rows from stroke_data table:
id	gender	age	hypertension	heart_disease	ever_married	work_type	Residence_type	avg_glucose_level	bmi	smoking_status	stroke
9046	Male	67.0	0	1	Yes	Private	Urban	228.69	36.6	formerly smoked	1
51676	Female	61.0	0	0	Yes	Self-employed	Rural	202.21	N/A	never smoked	1
31112	Male	80.0	0	1	Yes	Private	Rural	105.92	32.5	never smoked	1
*/
Thought:The "stro

There are 3353 persons who are married.
Final Answer: 3353

> Finished chain.


'3353'

In [ ]:
df.filter(df['ever_married'] == 'Yes').count()

3353